In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../../data/housing.csv")
print(df.shape)
df.head()

(20640, 10)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [3]:
from sklearn.model_selection import train_test_split
X = df.drop("median_house_value", axis=1)
y = df["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
def add_features(df):
    df = df.copy()
    df["rooms_per_household"] = df["total_rooms"] / df["households"]
    df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]
    df["population_per_household"] = df["population"] / df["households"]
    return df

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

num_cols = X_train.select_dtypes("number").columns.tolist()
num_cols += ["rooms_per_household", "bedrooms_per_room", "population_per_household"]
cat_cols = ["ocean_proximity"]

num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("sc", StandardScaler())])
cat_pipe = OneHotEncoder(handle_unknown="ignore")

preprocess = ColumnTransformer([("num", num_pipe, num_cols),
                                ("cat", cat_pipe, cat_cols)])

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

base = Pipeline([("feat", FunctionTransformer(add_features)),
                 ("prep", preprocess),
                 ("model", LinearRegression())])
base.fit(X_train, y_train)
pred = base.predict(X_test)
base_mae = mean_absolute_error(y_test, pred)
base_r2 = r2_score(y_test, pred)

In [7]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

for name, m in {"RF": RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42),
                "GB": GradientBoostingRegressor(random_state=42)}.items():
    pipe = Pipeline([("feat", FunctionTransformer(add_features)),
                     ("prep", preprocess), ("model", m)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
    print(name, -scores.mean())

RF 32869.88132425463
GB 36428.42508292922


In [18]:
import sys
sys.path.append("../..")
from src.features import add_features


from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.ensemble import RandomForestRegressor



num_cols = X_train.select_dtypes("number").columns.tolist()
num_cols += ["rooms_per_household", "bedrooms_per_room", "population_per_household"]
cat_cols = ["ocean_proximity"]

num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("sc", StandardScaler())])

preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

best_pipe = Pipeline([
    ("feat", FunctionTransformer(add_features)),
    ("prep", preprocess),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])

ModuleNotFoundError: No module named 'src.features'

In [10]:
from sklearn.model_selection import RandomizedSearchCV

params = {"model__n_estimators": [200, 400, 600],
          "model__max_depth": [None, 10, 20],
          "model__min_samples_leaf": [1, 2, 4]}

search = RandomizedSearchCV(best_pipe, params, n_iter=10, cv=5,
                            scoring="neg_mean_absolute_error",
                            n_jobs=-1, random_state=42)
search.fit(X_train, y_train)
print(search.best_params_)
print(-search.best_score_)

{'model__n_estimators': 200, 'model__min_samples_leaf': 2, 'model__max_depth': 20}
32793.73845019109


In [11]:
from sklearn.model_selection import RandomizedSearchCV
params = {"model__n_estimators": [200, 400, 600],
          "model__max_depth": [None, 10, 20],
          "model__min_samples_leaf": [1, 2, 4]}
search = RandomizedSearchCV(best_pipe, params, n_iter=10, cv=5,
                            scoring="neg_mean_absolute_error", n_jobs=-1, random_state=42)
search.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_distributions,"{'model__max_depth': [None, 10, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__n_estimators': [200, 400, ...]}"
,n_iter,10
,scoring,'neg_mean_absolute_error'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [12]:
final_pred = search.best_estimator_.predict(X_test)
final_mae = mean_absolute_error(y_test, final_pred)
final_r2 = r2_score(y_test, final_pred)
improvement = (base_mae - final_mae) / base_mae * 100
print(final_r2, final_mae, f"{improvement:.1f}% MAE reduction")

0.8130021647103304 31811.201688800164 35.9% MAE reduction


In [14]:
import joblib, os

os.makedirs("../../models", exist_ok=True)
joblib.dump(search.best_estimator_, "../../models/model.pkl")
print("Saved:", os.path.abspath("../../models/model.pkl"))

Saved: c:\House_price_prediction\house-price-project\models\model.pkl
